# 🔬 Combined Experiment 1 & 2: Repeated LOPO-CV (10x Empirical Consistency Benchmark)
**Research Group**: Group A01 — Advanced Software Engineering (Master of Computer Science, ITS)  
**Benchmark Objective**: Comprehensively evaluate the predictive accuracy, stochastic stability, and empirical consistency of the **Tri-Paradigm Data Cleaning Framework (C1: Row Discarding vs. C2: Project Discarding vs. C3: Group-Aware Median Imputation)** across 72 Factorial Machine Learning Pipelines over **10 Independent Stochastic Seeds** (Total: **16,080 Evaluated Model Folds**).  
**Validation Protocol**: Leave-One-Project-Out Cross-Validation (LOPO-CV, Zero Data Leakage) repeated 10 times (`[42, 100, 2024, 777, 999, 123, 456, 789, 2025, 2026]`).

## 1. Environment & Hardware Diagnostics (RTX 4050 CUDA Acceleration)

In [ ]:
import sys
import os
from pathlib import Path

src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import check_gpu
check_gpu.check_nvidia_smi()
check_gpu.check_python_environment()
check_gpu.check_xgboost_gpu()

## 2. Ingestion & Preliminary Comparison of Tri-Paradigm Datasets (C1, C2, C3)

In [ ]:
from data_loader import load_dataset

df_c1, feat_c1, proj_c1 = load_dataset('C1')
df_c2, feat_c2, proj_c2 = load_dataset('C2')
df_c3, feat_c3, proj_c3 = load_dataset('C3')

print("\n[PRELIMINARY DATASET COMPARISON]:")
print(f"  C1 (Row-Level Discarding)     : {len(df_c1):,} rows | {len(proj_c1)} projects | {df_c1['flaky'].sum()} flaky ({df_c1['flaky'].mean()*100:.2f}%)")
print(f"  C2 (Project-Level Discarding) : {len(df_c2):,} rows | {len(proj_c2)} projects | {df_c2['flaky'].sum()} flaky ({df_c2['flaky'].mean()*100:.2f}%)")
print(f"  C3 (Group-Aware Imputation)   : {len(df_c3):,} rows | {len(proj_c3)} projects | {df_c3['flaky'].sum()} flaky ({df_c3['flaky'].mean()*100:.2f}%)")

## 3. Repeated LOPO-CV Execution (Full 72 Pipelines x 10 Random Seeds = 16,080 Model Folds)
Executes all 72 factorial combinations (C1/C2/C3 x S1/S2/S3 x B1/B2/B3/B4 x M1/M2) across 10 independent stochastic random seeds.
- If cached benchmark records (`results/all_seed_pipeline_runs.csv`) exist on disk, this cell loads them instantly.
- To re-execute all 16,080 model folds from scratch, set `force_rerun = True` (~3.5 - 4 hours with GPU acceleration).

In [ ]:
import pandas as pd
from pipeline_builder import get_pipeline_specs
from repeated_lopo_runner import run_repeated_lopo_experiments, DEFAULT_SEEDS_10

all_specs = get_pipeline_specs()
runs_file = Path("../results/all_seed_pipeline_runs.csv")

# Execution control parameter:
force_rerun = False  # Set to True to re-execute all 16,080 model folds

if not runs_file.exists() or force_rerun:
    print(f"🚀 Launching full repeated LOPO-CV across {len(all_specs)} pipelines over 10 seeds...")
    results_full = run_repeated_lopo_experiments(
        pipelines=all_specs,
        seeds=DEFAULT_SEEDS_10,
        output_dir="../results",
        use_gpu=True
    )
    df_runs = results_full["all_runs"]
else:
    print(f"[OK] Cached 10-seed repetition results discovered: {runs_file.resolve()}")
    df_runs = pd.read_csv(runs_file)
    print(f"     Total Evaluations: {len(df_runs):,} pipeline runs ({df_runs['Pipeline_ID'].nunique()} pipelines x {df_runs['Seed'].nunique()} seeds)")

display(df_runs.head(10))

## 4. Empirical Consistency & Stability Analysis
Computes Sample Mean ($\mu$), Standard Deviation ($\sigma$), Coefficient of Variation ($CV = \frac{\sigma}{\mu} \times 100\%$), Min-Max performance ranges, and 95% Confidence Intervals for every pipeline configuration and experimental factor.

In [ ]:
from consistency_analyzer import analyze_pipeline_consistency

df_cons = analyze_pipeline_consistency(
    all_runs_csv="../results/all_seed_pipeline_runs.csv",
    output_dir="../results"
)

cols_show = [
    "Pipeline_ID", "Cleaning", "Scaling", "Imbalance", "Model", "Num_Seeds",
    "Mean_F1_Mean", "Mean_F1_Std", "Mean_F1_CV_%", "Mean_F1_CI95",
    "Mean_Recall_Mean", "Mean_Precision_Mean", "Mean_PR_AUC_Mean"
]
display(df_cons[[c for c in cols_show if c in df_cons.columns]].head(15))

## 5. Statistical Significance Testing (Friedman Omnibus, Nemenyi CD & Kruskal-Wallis)

In [ ]:
from statistical_tests import run_friedman_nemenyi_test, run_cleaning_comparison_test

print("=== FRIEDMAN OMNIBUS TEST (72 FACTORIAL PIPELINES) ===")
run_friedman_nemenyi_test("../results/summary_metrics_72_pipelines.csv", metric="Mean_F1")

print("\n=== KRUSKAL-WALLIS TEST: TRI-PARADIGM CLEANING COMPARISON (C1 vs C2 vs C3) ===")
run_cleaning_comparison_test("../results/summary_metrics_72_pipelines.csv")

## 6. Publication-Grade Visualizations (300 DPI)
Generates 6 IEEE/ACM publication-standard high-resolution figures into `results/plots/` and renders primary figures inline.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from visualizer import generate_all_visualizations

# 1. Generate all 6 publication-grade figures (300 DPI)
generate_all_visualizations(results_dir="../results", plots_dir="../results/plots")

# 2. Render primary figures inline
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(mpimg.imread("../results/plots/01_boxplot_tri_paradigm_cleaning.png"))
axes[0].axis("off")
axes[0].set_title("Figure 1: Tri-Paradigm Cleaning Benchmark Comparison", fontweight="bold", fontsize=12)

axes[1].imshow(mpimg.imread("../results/plots/03_top10_consistent_pipelines_ci95.png"))
axes[1].axis("off")
axes[1].set_title("Figure 3: Top 10 High-Performing Pipelines (95% CI)", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))
axes2[0].imshow(mpimg.imread("../results/plots/04_stability_cv_distribution.png"))
axes2[0].axis("off")
axes2[0].set_title("Figure 4: Empirical Stability Benchmark (CV%)", fontweight="bold", fontsize=12)

axes2[1].imshow(mpimg.imread("../results/plots/05_f1_vs_recall_tradeoff.png"))
axes2[1].axis("off")
axes2[1].set_title("Figure 5: Performance Trade-off (F1 vs. Recall)", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()